In [ ]:
!pip install transformers datasets torch accelerate scikit-learn tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import json, random, os
from tqdm import tqdm
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
import random
RAW_DATA = [

    # =========================================================================
    # PYTHON — SECURITY (high severity)
    # =========================================================================
    {
        "code": "password = 'abc123'",
        "comment": "Hardcoded credentials detected. Move secrets to environment variables: os.environ.get('PASSWORD').",
        "reward": 0.97, "category": "Security", "severity": "high"
    },
    {
        "code": "password = 'abc123'",
        "comment": "Maybe rename the variable.",
        "reward": 0.08, "category": "Security", "severity": "high"
    },
    {
        "code": "password = 'abc123'",
        "comment": "Don't hardcode passwords. Use environment variables instead.",
        "reward": 0.65, "category": "Security", "severity": "high"
    },
    {
        "code": "def get_user(id):\n    return db.query('SELECT * FROM users WHERE id=' + str(id))",
        "comment": "SQL injection risk! Use parameterized queries: db.query('SELECT * FROM users WHERE id=?', (id,))",
        "reward": 0.99, "category": "Security", "severity": "high"
    },
    {
        "code": "def get_user(id):\n    return db.query('SELECT * FROM users WHERE id=' + str(id))",
        "comment": "Queries should be more efficient.",
        "reward": 0.04, "category": "Security", "severity": "high"
    },
    {
        "code": "def get_user(id):\n    return db.query('SELECT * FROM users WHERE id=' + str(id))",
        "comment": "This directly concatenates user input into a query which allows SQL injection. Use parameterized queries.",
        "reward": 0.72, "category": "Security", "severity": "high"
    },
    {
        "code": "token = '8f14e45f'\nheaders = {'Authorization': token}",
        "comment": "API token is hardcoded in source. Load it from an environment variable or secrets manager to prevent credential leaks.",
        "reward": 0.96, "category": "Security", "severity": "high"
    },
    {
        "code": "token = '8f14e45f'\nheaders = {'Authorization': token}",
        "comment": "Token looks short.",
        "reward": 0.03, "category": "Security", "severity": "high"
    },
    {
        "code": "exec(user_input)",
        "comment": "Never pass unsanitized user input to exec(). This allows arbitrary code execution. Validate and whitelist inputs instead.",
        "reward": 0.98, "category": "Security", "severity": "high"
    },
    {
        "code": "exec(user_input)",
        "comment": "This runs code.",
        "reward": 0.02, "category": "Security", "severity": "high"
    },
    {
        "code": "import pickle\nobj = pickle.loads(data)",
        "comment": "Deserializing untrusted data with pickle can execute arbitrary code. Use a safe format like JSON instead.",
        "reward": 0.95, "category": "Security", "severity": "high"
    },
    {
        "code": "import pickle\nobj = pickle.loads(data)",
        "comment": "Consider using a different serialization format.",
        "reward": 0.50, "category": "Security", "severity": "high"
    },
    {
        "code": "def hash_pass(p):\n    return md5(p.encode()).hexdigest()",
        "comment": "MD5 is cryptographically broken. Use bcrypt or argon2 for password hashing: `bcrypt.hashpw(p, bcrypt.gensalt())`.",
        "reward": 0.97, "category": "Security", "severity": "high"
    },
    {
        "code": "def hash_pass(p):\n    return md5(p.encode()).hexdigest()",
        "comment": "Hashing is done here.",
        "reward": 0.02, "category": "Security", "severity": "high"
    },
    {
        "code": "app.run(debug=True, host='0.0.0.0')",
        "comment": "Running Flask in debug mode on all interfaces exposes the Werkzeug debugger publicly. Disable debug in production and bind to localhost.",
        "reward": 0.94, "category": "Security", "severity": "high"
    },
    {
        "code": "app.run(debug=True, host='0.0.0.0')",
        "comment": "Debug mode is enabled.",
        "reward": 0.55, "category": "Security", "severity": "high"
    },

    # =========================================================================
    # PYTHON — ERROR HANDLING (high/medium severity)
    # =========================================================================
    {
        "code": "try:\n    x = int(input())\nexcept:\n    pass",
        "comment": "Bare except swallows all exceptions including KeyboardInterrupt. Catch specific exceptions like ValueError.",
        "reward": 0.91, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "try:\n    x = int(input())\nexcept:\n    pass",
        "comment": "You should handle errors.",
        "reward": 0.10, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "try:\n    x = int(input())\nexcept:\n    pass",
        "comment": "The bare except is problematic — you should catch specific exceptions.",
        "reward": 0.60, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "open('file.txt').read()",
        "comment": "File handle is never closed. Use a context manager: `with open('file.txt') as f: data = f.read()`",
        "reward": 0.93, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "open('file.txt').read()",
        "comment": "Reading a file here.",
        "reward": 0.07, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "def divide(a, b):\n    return a / b",
        "comment": "No guard against division by zero. Add `if b == 0: raise ValueError('Divisor cannot be zero')` before returning.",
        "reward": 0.90, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "def divide(a, b):\n    return a / b",
        "comment": "Division is performed.",
        "reward": 0.03, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "def divide(a, b):\n    return a / b",
        "comment": "Should handle division by zero.",
        "reward": 0.52, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "response = requests.get(url)\ndata = response.json()",
        "comment": "No error handling for failed HTTP requests. Check `response.raise_for_status()` and wrap in try/except requests.RequestException.",
        "reward": 0.92, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "response = requests.get(url)\ndata = response.json()",
        "comment": "This makes an HTTP request.",
        "reward": 0.04, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "value = my_dict[key]",
        "comment": "Direct dict access raises KeyError if key is missing. Use `my_dict.get(key)` or `my_dict.get(key, default)` for safe access.",
        "reward": 0.88, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "value = my_dict[key]",
        "comment": "Key access could fail.",
        "reward": 0.48, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "value = my_dict[key]",
        "comment": "Dictionary is accessed.",
        "reward": 0.04, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "except Exception as e:\n    print(e)\n    pass",
        "comment": "Catching Exception broadly and silencing it hides bugs. Log with logging.exception(e) and only catch what you can handle.",
        "reward": 0.87, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "except Exception as e:\n    print(e)\n    pass",
        "comment": "Exception is printed.",
        "reward": 0.05, "category": "Error Handling", "severity": "medium"
    },

    # =========================================================================
    # PYTHON — BEST PRACTICE (low/medium severity)
    # =========================================================================
    {
        "code": "def add(a, b):\n    return a+b",
        "comment": "Consider adding type hints and a docstring. E.g.: def add(a: int, b: int) -> int:",
        "reward": 0.92, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def add(a, b):\n    return a+b",
        "comment": "This code is bad.",
        "reward": 0.05, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def add(a, b):\n    return a+b",
        "comment": "Adding type hints would improve this function for readability.",
        "reward": 0.55, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "import *\nfrom module import *",
        "comment": "Wildcard imports pollute the namespace and make it hard to trace where names come from. Import explicitly.",
        "reward": 0.82, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "import *\nfrom module import *",
        "comment": "Imports look OK to me.",
        "reward": 0.03, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "global counter\ncounter += 1",
        "comment": "Avoid global state. Encapsulate counter in a class or pass it as a parameter to keep functions pure.",
        "reward": 0.86, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "global counter\ncounter += 1",
        "comment": "Counter is being incremented.",
        "reward": 0.03, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "def process(items):\n    l = []\n    for i in items:\n        l.append(i)\n    return l",
        "comment": "Single-letter variable names like `l` are ambiguous (looks like 1). Use descriptive names and simplify: `return list(items)`.",
        "reward": 0.88, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def process(items):\n    l = []\n    for i in items:\n        l.append(i)\n    return l",
        "comment": "Avoid using 'l' as a variable name — it is ambiguous with the digit '1'. Use a descriptive name.",
        "reward": 0.65, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def process(items):\n    l = []\n    for i in items:\n        l.append(i)\n    return l",
        "comment": "This function returns a list.",
        "reward": 0.04, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "class myClass:\n    pass",
        "comment": "Class names should follow PascalCase per PEP 8. Rename to `MyClass`.",
        "reward": 0.85, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "class myClass:\n    pass",
        "comment": "Class naming could be better.",
        "reward": 0.45, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "class myClass:\n    pass",
        "comment": "Empty class defined.",
        "reward": 0.04, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "from datetime import datetime\nnow = datetime.now()",
        "comment": "datetime.now() returns naive datetime (no timezone). Use datetime.now(timezone.utc) to avoid timezone-related bugs.",
        "reward": 0.89, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "from datetime import datetime\nnow = datetime.now()",
        "comment": "Gets current datetime.",
        "reward": 0.04, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "PI = 3.14",
        "comment": "Use `math.pi` for the mathematical constant pi instead of a less precise literal to avoid subtle floating-point errors.",
        "reward": 0.84, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "PI = 3.14",
        "comment": "Pi value assigned.",
        "reward": 0.03, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def is_even(n):\n    if n % 2 == 0:\n        return True\n    else:\n        return False",
        "comment": "Simplify to `return n % 2 == 0` — the comparison already returns a boolean.",
        "reward": 0.87, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def is_even(n):\n    if n % 2 == 0:\n        return True\n    else:\n        return False",
        "comment": "Function checks if even.",
        "reward": 0.04, "category": "Best Practice", "severity": "low"
    },

    # =========================================================================
    # PYTHON — READABILITY (low/medium severity)
    # =========================================================================
    {
        "code": "for i in range(len(lst)):\n    print(lst[i])",
        "comment": "Use enumerate or direct iteration: `for item in lst: print(item)` — avoids index-based access.",
        "reward": 0.88, "category": "Readability", "severity": "medium"
    },
    {
        "code": "for i in range(len(lst)):\n    print(lst[i])",
        "comment": "Looks fine.",
        "reward": 0.02, "category": "Readability", "severity": "low"
    },
    {
        "code": "for i in range(len(data)):\n    process(data[i])",
        "comment": "Direct iteration is better: `for item in data: process(item)`",
        "reward": 0.62, "category": "Readability", "severity": "low"
    },
    {
        "code": "result = []\nfor x in data:\n    if x > 0:\n        result.append(x*2)",
        "comment": "This can be a one-liner list comprehension: `result = [x*2 for x in data if x > 0]`",
        "reward": 0.85, "category": "Readability", "severity": "low"
    },
    {
        "code": "result = []\nfor x in data:\n    if x > 0:\n        result.append(x*2)",
        "comment": "This loop could be improved somehow.",
        "reward": 0.06, "category": "Readability", "severity": "low"
    },
    {
        "code": "result = []\nfor x in data:\n    result.append(x*2)",
        "comment": "A list comprehension could make this shorter.",
        "reward": 0.50, "category": "Readability", "severity": "low"
    },
    {
        "code": "if x == True:",
        "comment": "Avoid comparing to True/False with ==. Use `if x:` directly for idiomatic Python.",
        "reward": 0.80, "category": "Readability", "severity": "low"
    },
    {
        "code": "if x == True:",
        "comment": "Boolean comparison.",
        "reward": 0.04, "category": "Readability", "severity": "low"
    },
    {
        "code": "def f(x):\n    return x**2 + 2*x + 1",
        "comment": "This computes (x+1) squared. You can simplify to `return (x + 1) ** 2` for clarity.",
        "reward": 0.78, "category": "Readability", "severity": "low"
    },
    {
        "code": "results = {}\nfor k in keys:\n    results[k] = compute(k)",
        "comment": "Consider using a dict comprehension: `results = {k: compute(k) for k in keys}`",
        "reward": 0.83, "category": "Readability", "severity": "low"
    },
    {
        "code": "x = x + 1",
        "comment": "Use augmented assignment: `x += 1` for clarity and consistency.",
        "reward": 0.75, "category": "Readability", "severity": "low"
    },
    {
        "code": "x = x + 1",
        "comment": "This increments x.",
        "reward": 0.01, "category": "Readability", "severity": "low"
    },
    {
        "code": "name='Alice';age=30;city='NY'",
        "comment": "Multiple statements on one line hurt readability. Place each assignment on its own line per PEP 8.",
        "reward": 0.84, "category": "Readability", "severity": "low"
    },
    {
        "code": "name='Alice';age=30;city='NY'",
        "comment": "Variables are defined.",
        "reward": 0.03, "category": "Readability", "severity": "low"
    },
    {
        "code": "s = ''\nfor w in words:\n    s += w + ' '",
        "comment": "String concatenation in a loop is O(n^2). Use `s = ' '.join(words)` instead for O(n) performance.",
        "reward": 0.91, "category": "Readability", "severity": "medium"
    },
    {
        "code": "s = ''\nfor w in words:\n    s += w + ' '",
        "comment": "String is built in a loop.",
        "reward": 0.05, "category": "Readability", "severity": "medium"
    },
    {
        "code": "if not x == None:",
        "comment": "Use `if x is not None:` — comparing to None with == is unconventional and PEP 8 recommends identity checks.",
        "reward": 0.86, "category": "Readability", "severity": "low"
    },
    {
        "code": "if not x == None:",
        "comment": "Checks for None.",
        "reward": 0.04, "category": "Readability", "severity": "low"
    },

    # =========================================================================
    # PYTHON — EFFICIENCY (medium severity)
    # =========================================================================
    {
        "code": "def process(data):\n    for i in range(len(data)):\n        data[i] = data[i] * 2\n    return data",
        "comment": "Use a list comprehension or map(): `return [x * 2 for x in data]` — more Pythonic and efficient.",
        "reward": 0.87, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "time.sleep(0.001)\nwhile not done:",
        "comment": "Busy-waiting even with sleep is inefficient. Use threading.Event().wait() or asyncio for proper blocking.",
        "reward": 0.90, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "if item in list(my_set):",
        "comment": "Converting a set to a list for membership testing is O(n). Use `if item in my_set:` directly for O(1) lookup.",
        "reward": 0.92, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "if item in list(my_set):",
        "comment": "Membership is checked.",
        "reward": 0.03, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "nums = []\nfor i in range(1000000):\n    nums.append(i)",
        "comment": "For large ranges, use a generator expression or `list(range(1000000))` which is far faster than appending in a loop.",
        "reward": 0.88, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "nums = []\nfor i in range(1000000):\n    nums.append(i)",
        "comment": "Large list is built.",
        "reward": 0.04, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "squared = list(map(lambda x: x**2, numbers))",
        "comment": "A list comprehension is more readable here: `squared = [x**2 for x in numbers]`.",
        "reward": 0.79, "category": "Efficiency", "severity": "low"
    },
    {
        "code": "squared = list(map(lambda x: x**2, numbers))",
        "comment": "Squares the numbers.",
        "reward": 0.04, "category": "Efficiency", "severity": "low"
    },
    {
        "code": "total = 0\nfor n in numbers:\n    total = total + n",
        "comment": "Use `sum(numbers)` — built-in functions are faster and more readable than manual accumulation loops.",
        "reward": 0.90, "category": "Efficiency", "severity": "low"
    },
    {
        "code": "total = 0\nfor n in numbers:\n    total = total + n",
        "comment": "Sums the numbers.",
        "reward": 0.03, "category": "Efficiency", "severity": "low"
    },
    {
        "code": "df2 = df\ndf2['col'] = 1",
        "comment": "This modifies the original DataFrame via reference. Use `df2 = df.copy()` to avoid unintended side effects.",
        "reward": 0.93, "category": "Efficiency", "severity": "high"
    },
    {
        "code": "df2 = df\ndf2['col'] = 1",
        "comment": "Column is set.",
        "reward": 0.03, "category": "Efficiency", "severity": "high"
    },

    # =========================================================================
    # PYTHON — STYLE (low severity)
    # =========================================================================
    {
        "code": "def myFunction(myParam):\n    myVar=myParam+1\n    return myVar",
        "comment": "Python uses snake_case for functions and variables per PEP 8. Rename to `my_function`, `my_param`, `my_var`.",
        "reward": 0.85, "category": "Style", "severity": "low"
    },
    {
        "code": "def myFunction(myParam):\n    myVar=myParam+1\n    return myVar",
        "comment": "Naming style could be improved.",
        "reward": 0.45, "category": "Style", "severity": "low"
    },
    {
        "code": "def myFunction(myParam):\n    myVar=myParam+1\n    return myVar",
        "comment": "The function returns a value.",
        "reward": 0.04, "category": "Style", "severity": "low"
    },
    {
        "code": "x=1\ny=2\nz=x+y",
        "comment": "Missing spaces around operators per PEP 8. Write `x = 1`, `y = 2`, `z = x + y`.",
        "reward": 0.81, "category": "Style", "severity": "low"
    },
    {
        "code": "x=1\ny=2\nz=x+y",
        "comment": "Code works.",
        "reward": 0.02, "category": "Style", "severity": "low"
    },
    {
        "code": "# TODO: fix this later\ndef process(): pass",
        "comment": "TODO comment left in code. Resolve it or create a tracked issue so it doesn't get forgotten.",
        "reward": 0.77, "category": "Style", "severity": "low"
    },
    {
        "code": "# TODO: fix this later\ndef process(): pass",
        "comment": "There is a comment here.",
        "reward": 0.03, "category": "Style", "severity": "low"
    },
    {
        "code": "MAGIC = 42\nif x > MAGIC:",
        "comment": "If MAGIC is a meaningful threshold, document its purpose with a comment or use a more descriptive constant name.",
        "reward": 0.76, "category": "Style", "severity": "low"
    },

    # =========================================================================
    # JAVASCRIPT — SECURITY
    # =========================================================================
    {
        "code": "element.innerHTML = userInput;",
        "comment": "Directly assigning user input to innerHTML enables XSS attacks. Use textContent or sanitize with DOMPurify first.",
        "reward": 0.98, "category": "Security", "severity": "high"
    },
    {
        "code": "element.innerHTML = userInput;",
        "comment": "Content is being set.",
        "reward": 0.03, "category": "Security", "severity": "high"
    },
    {
        "code": "element.innerHTML = userInput;",
        "comment": "Avoid innerHTML with untrusted data to prevent XSS.",
        "reward": 0.63, "category": "Security", "severity": "high"
    },
    {
        "code": "const apiKey = 'sk-prod-abc123';\nfetch(`/api?key=${apiKey}`);",
        "comment": "API keys must never be embedded in client-side JS — they are visible to all users. Move this to a backend proxy.",
        "reward": 0.97, "category": "Security", "severity": "high"
    },
    {
        "code": "const apiKey = 'sk-prod-abc123';\nfetch(`/api?key=${apiKey}`);",
        "comment": "Key is hardcoded.",
        "reward": 0.50, "category": "Security", "severity": "high"
    },
    {
        "code": "eval(userCode);",
        "comment": "eval() with user-controlled input is a critical security vulnerability. Use a sandboxed alternative or a proper parser.",
        "reward": 0.99, "category": "Security", "severity": "high"
    },
    {
        "code": "eval(userCode);",
        "comment": "Eval is used.",
        "reward": 0.04, "category": "Security", "severity": "high"
    },

    # =========================================================================
    # JAVASCRIPT — ERROR HANDLING
    # =========================================================================
    {
        "code": "async function load() {\n  const res = await fetch(url);\n  const data = await res.json();\n}",
        "comment": "No try/catch for the async fetch. Wrap in try/catch and handle network errors and non-OK HTTP statuses explicitly.",
        "reward": 0.93, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "async function load() {\n  const res = await fetch(url);\n  const data = await res.json();\n}",
        "comment": "Async fetch done.",
        "reward": 0.04, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "JSON.parse(rawString);",
        "comment": "JSON.parse throws SyntaxError on invalid input. Wrap it in a try/catch block to handle malformed JSON gracefully.",
        "reward": 0.89, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "JSON.parse(rawString);",
        "comment": "Parses JSON.",
        "reward": 0.04, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "promise.then(result => use(result));",
        "comment": "Promise has no .catch() handler. Unhandled rejections will be silently swallowed. Add `.catch(err => console.error(err))`.",
        "reward": 0.91, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "promise.then(result => use(result));",
        "comment": "Promise resolves.",
        "reward": 0.03, "category": "Error Handling", "severity": "high"
    },

    # =========================================================================
    # JAVASCRIPT — READABILITY / BEST PRACTICE
    # =========================================================================
    {
        "code": "var x = 10;",
        "comment": "Use `const` or `let` instead of `var`. `var` has function scope and hoisting which leads to subtle bugs.",
        "reward": 0.88, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "var x = 10;",
        "comment": "Variable declared.",
        "reward": 0.03, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "if (x == null) {}",
        "comment": "Use `===` for strict equality. `==` coerces types which can cause unexpected behavior (e.g., 0 == false is true).",
        "reward": 0.87, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "if (x == null) {}",
        "comment": "Null check is here.",
        "reward": 0.04, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "const items = arr.filter(x => x !== null).map(x => x.name);",
        "comment": "Clean chaining of filter and map — good. Consider adding a comment if the nulls being filtered have business significance.",
        "reward": 0.76, "category": "Readability", "severity": "low"
    },
    {
        "code": "console.log('debug', data);",
        "comment": "Debug console.log left in code. Remove before merging or replace with a proper logging utility.",
        "reward": 0.82, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "console.log('debug', data);",
        "comment": "Logs data.",
        "reward": 0.04, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "function greet(name) {\n  return 'Hello, ' + name + '!';\n}",
        "comment": "Use template literals for cleaner string interpolation: `return `Hello, ${name}!``.",
        "reward": 0.83, "category": "Readability", "severity": "low"
    },
    {
        "code": "function greet(name) {\n  return 'Hello, ' + name + '!';\n}",
        "comment": "Returns a greeting.",
        "reward": 0.03, "category": "Readability", "severity": "low"
    },

    # =========================================================================
    # TYPESCRIPT — BEST PRACTICE / READABILITY
    # =========================================================================
    {
        "code": "function process(data: any) {\n  return data.value;\n}",
        "comment": "Using `any` defeats TypeScript's type safety. Define a proper interface: `interface Data { value: unknown }` and use it.",
        "reward": 0.93, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "function process(data: any) {\n  return data.value;\n}",
        "comment": "Accesses data value.",
        "reward": 0.03, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "function process(data: any) {\n  return data.value;\n}",
        "comment": "Avoid using `any` — define a proper type or interface.",
        "reward": 0.60, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "const result = obj as any as TargetType;",
        "comment": "Double casting via `any` bypasses type safety entirely. If you need this cast, add a comment explaining why and consider a type guard instead.",
        "reward": 0.91, "category": "Best Practice", "severity": "high"
    },
    {
        "code": "const result = obj as any as TargetType;",
        "comment": "Type cast applied.",
        "reward": 0.04, "category": "Best Practice", "severity": "high"
    },
    {
        "code": "type ID = string | number | null | undefined;",
        "comment": "Consider using a branded type or a stricter union if IDs are always present at usage sites, to avoid handling null/undefined everywhere.",
        "reward": 0.77, "category": "Best Practice", "severity": "low"
    },

    # =========================================================================
    # SQL — SECURITY / EFFICIENCY / BEST PRACTICE
    # =========================================================================
    {
        "code": "SELECT * FROM orders WHERE customer_id = '' + customer_id",
        "comment": "String concatenation in SQL creates an injection vulnerability. Use parameterized queries or an ORM.",
        "reward": 0.98, "category": "Security", "severity": "high"
    },
    {
        "code": "SELECT * FROM orders WHERE customer_id = '' + customer_id",
        "comment": "Query selects orders.",
        "reward": 0.04, "category": "Security", "severity": "high"
    },
    {
        "code": "SELECT * FROM users;",
        "comment": "Avoid SELECT * in production code. Specify only the columns you need to reduce data transfer and avoid breakage when schema changes.",
        "reward": 0.87, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "SELECT * FROM users;",
        "comment": "Selects all users.",
        "reward": 0.04, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "SELECT * FROM users;",
        "comment": "Should specify columns instead of using *.",
        "reward": 0.55, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "SELECT name FROM products WHERE name LIKE '%phone%';",
        "comment": "Leading wildcard in LIKE prevents index usage resulting in a full table scan. Consider a full-text search index for this query.",
        "reward": 0.93, "category": "Efficiency", "severity": "high"
    },
    {
        "code": "SELECT name FROM products WHERE name LIKE '%phone%';",
        "comment": "Searches products by name.",
        "reward": 0.04, "category": "Efficiency", "severity": "high"
    },
    {
        "code": "DELETE FROM logs;",
        "comment": "DELETE without a WHERE clause removes all rows. Add a condition or use TRUNCATE if you intentionally want to clear the table.",
        "reward": 0.96, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "DELETE FROM logs;",
        "comment": "Deletes logs.",
        "reward": 0.04, "category": "Error Handling", "severity": "high"
    },

    # =========================================================================
    # JAVA — ERROR HANDLING / BEST PRACTICE / SECURITY
    # =========================================================================
    {
        "code": "public String getUser(int id) {\n    return db.query(\"SELECT * FROM users WHERE id=\" + id);\n}",
        "comment": "SQL injection vulnerability via string concatenation. Use PreparedStatement: `ps.setInt(1, id)` instead.",
        "reward": 0.97, "category": "Security", "severity": "high"
    },
    {
        "code": "public String getUser(int id) {\n    return db.query(\"SELECT * FROM users WHERE id=\" + id);\n}",
        "comment": "Returns user by id.",
        "reward": 0.03, "category": "Security", "severity": "high"
    },
    {
        "code": "catch (Exception e) {}",
        "comment": "Empty catch block silently swallows exceptions. At minimum log with `e.printStackTrace()` and ideally handle or rethrow.",
        "reward": 0.94, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "catch (Exception e) {}",
        "comment": "Exception is caught.",
        "reward": 0.03, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "catch (Exception e) {}",
        "comment": "Should not swallow exceptions silently.",
        "reward": 0.58, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "String s = null;\nSystem.out.println(s.length());",
        "comment": "Calling .length() on a null String throws NullPointerException. Add a null check: `if (s != null)` or use `Objects.requireNonNull`.",
        "reward": 0.95, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "String s = null;\nSystem.out.println(s.length());",
        "comment": "Prints string length.",
        "reward": 0.04, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "for (int i=0; i<list.size(); i++) {\n    process(list.get(i));\n}",
        "comment": "Prefer enhanced for-loop: `for (Item item : list) { process(item); }` — cleaner and avoids index errors.",
        "reward": 0.84, "category": "Readability", "severity": "low"
    },
    {
        "code": "for (int i=0; i<list.size(); i++) {\n    process(list.get(i));\n}",
        "comment": "Iterates through list.",
        "reward": 0.04, "category": "Readability", "severity": "low"
    },
    {
        "code": "public class DataManager {\n    public static List<Object> cache = new ArrayList<>();\n}",
        "comment": "Public static mutable state is dangerous in multi-threaded environments. Make it private, add synchronization or use ConcurrentArrayList.",
        "reward": 0.91, "category": "Best Practice", "severity": "high"
    },
    {
        "code": "public class DataManager {\n    public static List<Object> cache = new ArrayList<>();\n}",
        "comment": "Cache is declared.",
        "reward": 0.04, "category": "Best Practice", "severity": "high"
    },

    # =========================================================================
    # GO — ERROR HANDLING / BEST PRACTICE
    # =========================================================================
    {
        "code": "result, _ := os.Open(filename)",
        "comment": "Ignoring the error return with _ is dangerous here. Check the error: `f, err := os.Open(filename); if err != nil { return err }`.",
        "reward": 0.95, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "result, _ := os.Open(filename)",
        "comment": "File is opened.",
        "reward": 0.03, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "result, _ := os.Open(filename)",
        "comment": "Error is being ignored which could cause issues.",
        "reward": 0.58, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "func process(data []int) []int {\n    result := make([]int, len(data))\n    for i, v := range data {\n        result[i] = v * 2\n    }\n    return result\n}",
        "comment": "Clean and idiomatic Go. Consider pre-allocating with capacity hint if data size is known and large, for a minor performance gain.",
        "reward": 0.79, "category": "Efficiency", "severity": "low"
    },
    {
        "code": "var wg sync.WaitGroup\ngo func() { task() }()",
        "comment": "Goroutine launched without wg.Add(1)/wg.Done(). Add `wg.Add(1)` before go and `defer wg.Done()` inside to prevent premature exits.",
        "reward": 0.94, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "var wg sync.WaitGroup\ngo func() { task() }()",
        "comment": "Goroutine is started.",
        "reward": 0.04, "category": "Error Handling", "severity": "high"
    },

    # =========================================================================
    # C++ — SECURITY / ERROR HANDLING / BEST PRACTICE
    # =========================================================================
    {
        "code": "char buf[64];\ngets(buf);",
        "comment": "`gets()` has no bounds checking and causes buffer overflows. Replace with `fgets(buf, sizeof(buf), stdin)` immediately.",
        "reward": 0.99, "category": "Security", "severity": "high"
    },
    {
        "code": "char buf[64];\ngets(buf);",
        "comment": "Reads user input.",
        "reward": 0.03, "category": "Security", "severity": "high"
    },
    {
        "code": "int* ptr = new int[100];\n// ... use ptr",
        "comment": "Memory allocated with new[] but no delete[] visible. This leaks memory. Use std::vector<int> or std::unique_ptr<int[]> instead.",
        "reward": 0.95, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "int* ptr = new int[100];\n// ... use ptr",
        "comment": "Array is allocated.",
        "reward": 0.03, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "int* ptr = new int[100];\n// ... use ptr",
        "comment": "Should probably free the allocated memory.",
        "reward": 0.55, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "void copy(char* dst, const char* src) {\n    strcpy(dst, src);\n}",
        "comment": "strcpy does not check bounds and can overflow dst. Use strncpy(dst, src, SIZE-1) or preferably std::string in C++.",
        "reward": 0.96, "category": "Security", "severity": "high"
    },
    {
        "code": "void copy(char* dst, const char* src) {\n    strcpy(dst, src);\n}",
        "comment": "Copies a string.",
        "reward": 0.04, "category": "Security", "severity": "high"
    },

    # =========================================================================
    # BASH / SHELL — SECURITY / BEST PRACTICE
    # =========================================================================
    {
        "code": 'eval "$user_input"',
        "comment": "eval with unvalidated user input allows arbitrary command execution. Never eval user-controlled strings in shell scripts.",
        "reward": 0.98, "category": "Security", "severity": "high"
    },
    {
        "code": 'eval "$user_input"',
        "comment": "Evaluates input.",
        "reward": 0.04, "category": "Security", "severity": "high"
    },
    {
        "code": "#!/bin/bash\nrm -rf $DIR/",
        "comment": 'If $DIR is empty or unset, this expands to `rm -rf /` destroying the filesystem. Quote the variable and check it: `[[ -n $DIR ]] && rm -rf "$DIR/"`.',
        "reward": 0.99, "category": "Security", "severity": "high"
    },
    {
        "code": "#!/bin/bash\nrm -rf $DIR/",
        "comment": "Removes directory.",
        "reward": 0.04, "category": "Security", "severity": "high"
    },
    {
        "code": "#!/bin/bash\nrm -rf $DIR/",
        "comment": "Dangerous if $DIR is empty — should quote the variable.",
        "reward": 0.65, "category": "Security", "severity": "high"
    },
    {
        "code": "cp file.txt /tmp/backup",
        "comment": "No error handling after cp. Add `|| { echo 'backup failed'; exit 1; }` to fail fast on copy errors.",
        "reward": 0.85, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "cp file.txt /tmp/backup",
        "comment": "File is copied.",
        "reward": 0.03, "category": "Error Handling", "severity": "medium"
    },

    # =========================================================================
    # REACT / FRONTEND
    # =========================================================================
    {
        "code": "useEffect(() => {\n  fetchData();\n});",
        "comment": "useEffect with no dependency array runs after every render. Add `[]` for mount-only or `[dep]` to control when it fires.",
        "reward": 0.94, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "useEffect(() => {\n  fetchData();\n});",
        "comment": "Fetches data.",
        "reward": 0.04, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "useEffect(() => {\n  fetchData();\n});",
        "comment": "Missing dependency array causes infinite renders.",
        "reward": 0.62, "category": "Error Handling", "severity": "high"
    },
    {
        "code": "function List({ items }) {\n  return items.map(i => <div>{i}</div>);\n}",
        "comment": "Each child in a list must have a unique `key` prop. Add `key={i.id}` or a stable unique identifier to avoid reconciliation bugs.",
        "reward": 0.93, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "function List({ items }) {\n  return items.map(i => <div>{i}</div>);\n}",
        "comment": "Renders a list.",
        "reward": 0.04, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "const [data, setData] = useState(null);\nsetData(newData);",
        "comment": "If `setData` is called inside an async callback after unmount, it causes a memory leak. Use an `isMounted` flag or AbortController to clean up.",
        "reward": 0.90, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "const [data, setData] = useState(null);\nsetData(newData);",
        "comment": "State is updated.",
        "reward": 0.03, "category": "Error Handling", "severity": "medium"
    },
    {
        "code": "import React from 'react';\nimport ReactDOM from 'react-dom';\nimport App from './App';\nimport utils from './utils';\nimport helpers from './helpers';\nimport config from './config';\nimport styles from './styles';",
        "comment": "Large number of imports is fine but consider grouping: third-party first, then local modules, separated by blank lines for readability.",
        "reward": 0.74, "category": "Readability", "severity": "low"
    },

    # =========================================================================
    # DOCKER / DEVOPS
    # =========================================================================
    {
        "code": "FROM ubuntu:latest",
        "comment": "Using `latest` tag is non-deterministic — the image can change unexpectedly. Pin to a specific version like `ubuntu:22.04` for reproducible builds.",
        "reward": 0.92, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "FROM ubuntu:latest",
        "comment": "Base image specified.",
        "reward": 0.04, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "FROM ubuntu:latest",
        "comment": "Should pin the image version instead of using latest.",
        "reward": 0.58, "category": "Best Practice", "severity": "medium"
    },
    {
        "code": "RUN apt-get update\nRUN apt-get install -y curl",
        "comment": "Separate RUN commands create extra layers. Combine into one: `RUN apt-get update && apt-get install -y curl && rm -rf /var/lib/apt/lists/*`.",
        "reward": 0.89, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "RUN apt-get update\nRUN apt-get install -y curl",
        "comment": "Installs curl.",
        "reward": 0.03, "category": "Efficiency", "severity": "medium"
    },
    {
        "code": "COPY . /app",
        "comment": "Copying the entire context may include sensitive files (.env, .git). Add a .dockerignore file to exclude them.",
        "reward": 0.88, "category": "Security", "severity": "medium"
    },
    {
        "code": "COPY . /app",
        "comment": "Files are copied.",
        "reward": 0.04, "category": "Security", "severity": "medium"
    },

    # =========================================================================
    # GOOD CODE — model must learn not to penalise correct patterns
    # =========================================================================
    {
        "code": "def add(a: int, b: int) -> int:\n    \"\"\"Return the sum of a and b.\"\"\"\n    return a + b",
        "comment": "Well-typed and documented function. No issues.",
        "reward": 0.95, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "def add(a: int, b: int) -> int:\n    \"\"\"Return the sum of a and b.\"\"\"\n    return a + b",
        "comment": "This code is bad and needs improvement.",
        "reward": 0.02, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "with open('file.txt') as f:\n    data = f.read()",
        "comment": "Correct use of context manager — file is properly closed on exit.",
        "reward": 0.93, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "with open('file.txt') as f:\n    data = f.read()",
        "comment": "Should add error handling for file not found.",
        "reward": 0.58, "category": "Error Handling", "severity": "low"
    },
    {
        "code": "with open('file.txt') as f:\n    data = f.read()",
        "comment": "File is read.",
        "reward": 0.04, "category": "Best Practice", "severity": "low"
    },
    {
        "code": "result = [x * 2 for x in data if x > 0]",
        "comment": "Clean list comprehension with filter — idiomatic Python. No issues.",
        "reward": 0.94, "category": "Readability", "severity": "low"
    },
    {
        "code": "result = [x * 2 for x in data if x > 0]",
        "comment": "Should use a for loop for clarity.",
        "reward": 0.05, "category": "Readability", "severity": "low"
    },
    {
        "code": "try:\n    value = int(user_input)\nexcept ValueError:\n    value = 0",
        "comment": "Specific exception handling for ValueError is correct here. Well done.",
        "reward": 0.92, "category": "Error Handling", "severity": "low"
    },
    {
        "code": "try:\n    value = int(user_input)\nexcept ValueError:\n    value = 0",
        "comment": "Should use bare except for safety.",
        "reward": 0.03, "category": "Error Handling", "severity": "low"
    },
    {
        "code": "password = os.environ.get('DB_PASSWORD')",
        "comment": "Correct approach — loading credentials from environment. No issues.",
        "reward": 0.95, "category": "Security", "severity": "low"
    },
    {
        "code": "password = os.environ.get('DB_PASSWORD')",
        "comment": "This is insecure and should use a different method.",
        "reward": 0.02, "category": "Security", "severity": "low"
    },
    {
        "code": "const query = 'SELECT id, name FROM users WHERE id = ?';\ndb.run(query, [userId]);",
        "comment": "Parameterized query correctly prevents SQL injection. Good practice.",
        "reward": 0.94, "category": "Security", "severity": "low"
    },
    {
        "code": "const query = 'SELECT id, name FROM users WHERE id = ?';\ndb.run(query, [userId]);",
        "comment": "Should use string interpolation for the query.",
        "reward": 0.02, "category": "Security", "severity": "low"
    },

    # =========================================================================
    # EDGE CASES — very short / trivial code
    # =========================================================================
    {
        "code": "x = 1",
        "comment": "This is a simple assignment. No issues in isolation.",
        "reward": 0.88, "category": "Style", "severity": "low"
    },
    {
        "code": "x = 1",
        "comment": "This is terrible code.",
        "reward": 0.02, "category": "Style", "severity": "low"
    },
    {
        "code": "print('hello')",
        "comment": "Simple print statement. No issues.",
        "reward": 0.85, "category": "Style", "severity": "low"
    },
    {
        "code": "print('hello')",
        "comment": "Should use logging instead of print in all cases.",
        "reward": 0.25, "category": "Style", "severity": "low"
    },
    {
        "code": "return None",
        "comment": "Explicit `return None` is fine, though bare `return` is conventional in Python when returning None.",
        "reward": 0.78, "category": "Style", "severity": "low"
    },

]


# =============================================================================
# SMART AUGMENTATION
# =============================================================================

PARAPHRASE_POOL = {
    "type hints": [
        ("Add type annotations to clarify expected argument and return types.", 0.0),
        ("Missing type annotations make this harder to use in larger codebases.", 0.05),
    ],
    "list comprehension": [
        ("A list comprehension is more idiomatic: `[... for x in data]`.", 0.0),
        ("Rewrite with a list comprehension for conciseness.", -0.05),
    ],
    "SQL injection": [
        ("This pattern is vulnerable to SQL injection — always use parameterized queries.", 0.02),
        ("Avoid building SQL strings from user input; use prepared statements.", 0.0),
    ],
    "bare except": [
        ("Catching all exceptions with `except:` hides bugs. Be specific.", 0.0),
        ("Replace bare except with `except ValueError:` or the relevant exception type.", 0.02),
    ],
    "context manager": [
        ("Use `with` statement to ensure the file is closed even if an exception occurs.", 0.0),
        ("Wrap file operations in a `with` block to avoid resource leaks.", 0.02),
    ],
    "environment variable": [
        ("Store sensitive values in environment variables, not source code.", 0.0),
        ("Secrets in code get committed to version control. Use os.environ instead.", 0.02),
    ],
}


def _reward_noise(reward: float, sigma: float = 0.03) -> float:
    """Add small noise but stay within the same quality tier."""
    tier_bounds = [(0.0, 0.3), (0.31, 0.69), (0.70, 1.0)]
    for lo, hi in tier_bounds:
        if lo <= reward <= hi:
            noisy = reward + random.gauss(0, sigma)
            return round(max(lo, min(hi, noisy)), 4)
    return reward


def augment_data(data: list, target_total: int = 1200) -> list:
    """
    Build an augmented dataset up to `target_total` samples.
    Uses paraphrase rewrites + noise rather than pure duplication.
    """
    augmented = list(data)  # start with all original samples

    paraphrase_keys = list(PARAPHRASE_POOL.keys())

    while len(augmented) < target_total:
        base = random.choice(data).copy()

        # Attempt paraphrase substitution
        applied = False
        random.shuffle(paraphrase_keys)
        for key in paraphrase_keys:
            if key.lower() in base["comment"].lower():
                alt_comment, delta = random.choice(PARAPHRASE_POOL[key])
                new_reward = _reward_noise(
                    max(0.0, min(1.0, base["reward"] + delta))
                )
                augmented.append({
                    "code":     base["code"],
                    "comment":  alt_comment,
                    "reward":   new_reward,
                    "category": base["category"],
                    "severity": base["severity"],
                })
                applied = True
                break

        if not applied:
            # No paraphrase match — add with reward noise only
            base["reward"] = _reward_noise(base["reward"])
            augmented.append(base)

    return augmented


TRAINING_DATA = augment_data(RAW_DATA, target_total=1200)
print(f"Unique raw samples : {len(RAW_DATA)}")
print(f"Total after augment: {len(TRAINING_DATA)}")

# Quick distribution check
high   = sum(1 for d in TRAINING_DATA if d["reward"] >= 0.70)
medium = sum(1 for d in TRAINING_DATA if 0.30 < d["reward"] < 0.70)
low    = sum(1 for d in TRAINING_DATA if d["reward"] <= 0.30)
print(f"High reward (≥0.7): {high}  Medium (0.3–0.7): {medium}  Low (≤0.3): {low}")

# ── CELL 4: Dataset ───────────────────────────────────────────
MODEL_NAME = "microsoft/codebert-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class CodeReviewDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item    = self.data[idx]
        text    = f"[CODE] {item['code']} [REVIEW] {item['comment']}"
        encoded = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "reward":         torch.tensor(item["reward"], dtype=torch.float),
            "category":       item.get("category", "Style"),
            "severity":       item.get("severity", "low"),
        }

# Train / Val split
random.shuffle(TRAINING_DATA)
split     = int(0.85 * len(TRAINING_DATA))
train_ds  = CodeReviewDataset(TRAINING_DATA[:split], tokenizer)
val_ds    = CodeReviewDataset(TRAINING_DATA[split:], tokenizer)

train_loader = DataLoader(train_ds, batch_size=8,  shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8,  shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

# ── CELL 5: Reward Model Architecture ────────────────────────
class CodeReviewRewardModel(nn.Module):
    """
    CodeBERT backbone + MLP head → scalar reward ∈ [0, 1]
    Architecture mirrors Wang et al. (2024) reward modelling setup.
    """
    def __init__(self, backbone_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone_name)
        hidden       = self.encoder.config.hidden_size  # 768 for codebert-base

        self.reward_head = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()           # output ∈ [0, 1]
        )

        # Category classifier (auxiliary head)
        self.categories = ["Readability", "Efficiency", "Best Practice",
                           "Error Handling", "Style", "Security"]
        self.cat_head   = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, len(self.categories))
        )

        # Severity classifier (auxiliary head)
        self.severities  = ["low", "medium", "high"]
        self.sev_head    = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, len(self.severities))
        )

    def forward(self, input_ids, attention_mask):
        out      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb  = out.last_hidden_state[:, 0, :]   # [CLS] token
        reward   = self.reward_head(cls_emb).squeeze(-1)
        cat_logits = self.cat_head(cls_emb)
        sev_logits = self.sev_head(cls_emb)
        return reward, cat_logits, sev_logits

model = CodeReviewRewardModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# ── CELL 6: Training Loop ─────────────────────────────────────
EPOCHS    = 5
LR        = 2e-5
WARMUP    = 0.1

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

cat_map = {"Readability": 0, "Efficiency": 1, "Best Practice": 2,
           "Error Handling": 3, "Style": 4, "Security": 5}
sev_map = {"low": 0, "medium": 1, "high": 2}

def compute_loss(reward_pred, reward_true, cat_logits, cat_true, sev_logits, sev_true):
    mse_loss = F.mse_loss(reward_pred, reward_true)
    cat_labels = torch.tensor([cat_map.get(c, 4) for c in cat_true], device=DEVICE)
    sev_labels = torch.tensor([sev_map.get(s, 0) for s in sev_true], device=DEVICE)
    cat_loss   = F.cross_entropy(cat_logits, cat_labels)
    sev_loss   = F.cross_entropy(sev_logits, sev_labels)
    # Weighted combination
    return 0.6 * mse_loss + 0.25 * cat_loss + 0.15 * sev_loss

best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "val_mae": []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for batch in pbar:
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        rewards    = batch["reward"].to(DEVICE)
        categories = batch["category"]
        severities = batch["severity"]

        optimizer.zero_grad()
        pred_reward, cat_logits, sev_logits = model(input_ids, attn_mask)
        loss = compute_loss(pred_reward, rewards, cat_logits, categories, sev_logits, severities)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    # ── Validate ──
    model.eval()
    val_losses, mae_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids  = batch["input_ids"].to(DEVICE)
            attn_mask  = batch["attention_mask"].to(DEVICE)
            rewards    = batch["reward"].to(DEVICE)
            categories = batch["category"]
            severities = batch["severity"]

            pred_reward, cat_logits, sev_logits = model(input_ids, attn_mask)
            loss = compute_loss(pred_reward, rewards, cat_logits, categories, sev_logits, severities)
            val_losses.append(loss.item())
            mae_list.append(torch.abs(pred_reward - rewards).mean().item())

    avg_train = np.mean(train_losses)
    avg_val   = np.mean(val_losses)
    avg_mae   = np.mean(mae_list)
    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_mae"].append(avg_mae)
    print(f"Epoch {epoch}: train_loss={avg_train:.4f}  val_loss={avg_val:.4f}  val_MAE={avg_mae:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "reward_model.pt")
        print(f"  ✅ Saved best model (val_loss={best_val_loss:.4f})")

print("\nTraining complete!")
print(f"Best val_loss: {best_val_loss:.4f}")

# ── CELL 7: Save tokenizer config ────────────────────────────
tokenizer.save_pretrained("./tokenizer_config")
config = {
    "model_name": MODEL_NAME,
    "categories": model.categories,
    "severities": model.severities,
    "max_len": 256,
    "hidden_size": model.encoder.config.hidden_size,
}
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Files saved:")
print("  reward_model.pt         ← load in VSCode backend")
print("  tokenizer_config/       ← tokenizer files")
print("  model_config.json       ← model metadata")
print("\nDownload all three to your VSCode project folder!")

# ── CELL 8: Quick inference test ─────────────────────────────
model.load_state_dict(torch.load("reward_model.pt", map_location=DEVICE))
model.eval()

test_samples = [
    ("def f(x): return x", "This function lacks a docstring. Add one explaining parameters and return type."),
    ("password='secret'", "Good variable naming."),
    ("try:\n  x=int(y)\nexcept:\n  pass", "Bare except swallows all exceptions. Catch ValueError specifically."),
]

print("\n── Inference Test ──")
for code, comment in test_samples:
    text = f"[CODE] {code} [REVIEW] {comment}"
    enc = tokenizer(text, return_tensors="pt", max_length=256, padding="max_length", truncation=True)
    with torch.no_grad():
        r, cat_l, sev_l = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    cat_idx = cat_l.argmax(-1).item()
    sev_idx = sev_l.argmax(-1).item()
    print(f"  Reward: {r.item():.3f}  Cat: {model.categories[cat_idx]}  Sev: {model.severities[sev_idx]}")
    print(f"  Code:    {code[:50]}")
    print(f"  Comment: {comment[:60]}")
    print()

Unique raw samples : 189
Total after augment: 1200
High reward (≥0.7): 596  Medium (0.3–0.7): 120  Low (≤0.3): 484


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Epoch 3: train_loss=0.0990  val_loss=0.0428  val_MAE=0.0718
  ✅ Saved best model (val_loss=0.0428)


Epoch 4/5 [Train]: 100%|██████████| 128/128 [00:52<00:00,  2.45it/s, loss=0.0656]


Epoch 4: train_loss=0.0496  val_loss=0.0254  val_MAE=0.0579
  ✅ Saved best model (val_loss=0.0254)


Epoch 5/5 [Train]: 100%|██████████| 128/128 [00:51<00:00,  2.47it/s, loss=0.0238]


Epoch 5: train_loss=0.0348  val_loss=0.0203  val_MAE=0.0536
  ✅ Saved best model (val_loss=0.0203)

Training complete!
Best val_loss: 0.0203
Files saved:
  reward_model.pt         ← load in VSCode backend
  tokenizer_config/       ← tokenizer files
  model_config.json       ← model metadata

Download all three to your VSCode project folder!

── Inference Test ──
  Reward: 0.356  Cat: Style  Sev: low
  Code:    def f(x): return x
  Comment: This function lacks a docstring. Add one explaining paramete

  Reward: 0.074  Cat: Security  Sev: high
  Code:    password='secret'
  Comment: Good variable naming.

  Reward: 0.767  Cat: Error Handling  Sev: high
  Code:    try:
  x=int(y)
except:
  pass
  Comment: Bare except swallows all exceptions. Catch ValueError specif



In [ ]:
from google.colab import files
files.download('reward_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>